# Experiments with DIFFI

With this notebook we want to test how many and which of the important features have been exploited by each isolation tree in a given isolation forest.

## Imports & Initializations

In [ ]:
import os
import numpy as np
import pickle as pkl 
import matplotlib.pyplot as plt 
%matplotlib inline
# from sklearn.ensemble import IsolationForest
# from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split
# import shap
# import diffi.interpretability_module as interp
from diffi.utils import *
import wandb
import odds_datasets
import seaborn as sns

In [ ]:
wandb.login()

In [ ]:
np.random.seed(0)

## Parameters

Parameters for the model

In [ ]:
num_trees = 100
# max_samples = 256   # for the dataset in ODDS is larger than the their size
n_forests = 10
percentile = 50
test_size = 0.0

## Functions definitions

In [ ]:
def log_model_config(contamination, max_samples):
    """
    Log the model configuration parameters to wandb.
    """
    config_data = [
        ["n_forests", n_forests],
        ["num_trees", num_trees],
        ["max_samples", max_samples],
        ["contamination", contamination],
    ]

    config_table = wandb.Table(data=config_data, columns=["Parameter", "Value"])

    wandb.log({"Configuration Table": config_table})

    # # table for n_forests and usage_threshold
    # data = [[f"Forest {i+1}", usage_threshold[i]] for i in range(n_forests)]
    # columns = ["Forest", "Usage Threshold"]

    # usage_threshold_table = wandb.Table(data=data, columns=columns)

    # # Log the table to wandb
    # wandb.log({"Usage Threshold Table": usage_threshold_table})

Plotting the feature importance values for each feature

In [ ]:
def log_feature_importance(feature_rank, fi_means, fi_std, og_model: bool):
    """
    Plot and log the feature importance to wandb.
    """
    
    plt.figure(figsize=(10, 5))
    plt.bar(range(len(feature_rank)), fi_means[feature_rank], yerr=fi_std[feature_rank])
    plt.xticks(range(len(feature_rank)), feature_rank)
    plt.xlabel('Feature index')
    plt.ylabel('Feature importance')
    if og_model:
        plt.title('Feature importance on original model')
        wandb.log({"original_model_feature_importance_image": wandb.Image(plt)})
    else:
        plt.title('Feature importance on new model')
        wandb.log({"new_model_feature_importance_image": wandb.Image(plt)})

    # plt.show()

Plotting the heatmap of feature importance for every trees 

In [ ]:
def log_feature_importance_heatmap(fi_diffi, for_inliers: bool):
    """
    Log the feature importance heatmap to wandb.
    """
    for i, forest in enumerate(fi_diffi):
        plt.figure(figsize=(12, 8))
        plt.plot(1, len(fi_diffi), i + 1)
        sns.heatmap(forest, cmap='viridis', cbar=True, vmin=0, vmax=0.16)
        plt.xlabel('Feature Index')
        plt.ylabel('Tree Index')
        # plt.tight_layout()
        if for_inliers:
            plt.title(f'Feature Importance Heatmaps for Inliers - Forest {i + 1}', fontsize=16)
            wandb.log({"feature_importance_heatmap_inliers": wandb.Image(plt)})
        else:
            plt.title(f'Feature Importance Heatmaps for Outliers - Forest {i + 1}', fontsize=16)
            wandb.log({"feature_importance_heatmap_outliers": wandb.Image(plt)})
            
        # plt.show()

Counting how many times the features are used in each Isolation Tree

In [ ]:
def feature_usage(features_per_forest, features):
    """
    Computing and plotting the feature usage in each tree and forest.
    """

    usage_per_forest = np.zeros((n_forests, len(features)), 
                                dtype=object)    # shape: (number of forests, number of features)
    usage_per_tree = np.zeros((n_forests, num_trees, len(features)), 
                            dtype=object)       # shape: (number of forests, number of trees, number of features)


    for i, feature in enumerate(features):                                              # for each feature                                     
        for j, forest in enumerate(features_per_forest):                                # for each forest
            for k, tree in enumerate(forest):                                           # for each tree
                # count the number of times feature i is used in tree k in forest j
                usage_per_tree[j, k, i] = np.sum([1 for f in tree if f == feature]) 
            # count the number of times feature i is used in forest j
            usage_per_forest[j, i] = np.sum(usage_per_tree[j, :, i])                          

    plt.figure(figsize=(12, 6))
    for i in range(usage_per_forest.shape[0]):
        plt.bar(range(usage_per_forest.shape[1]), usage_per_forest[i], alpha=0.5, label=f'Forest {i+1}')
    plt.xlabel('Feature Index')
    plt.ylabel('Usage Count')
    plt.title('Feature Usage Across Forests')
    plt.legend()
    plt.tight_layout()

    wandb.log({"feature_usage_image": wandb.Image(plt)})

    # plt.show()

    return usage_per_forest, usage_per_tree

Extract the counter for the `most_important_features`, used in each Isolation Tree of the Isolation Forest

In [ ]:
def most_important_feature_usage(most_important_features, usage_per_tree, features_per_forest):
    """
    Compute the usage of the most important features in each tree.
    """

    most_important_features_usage = np.zeros((len(most_important_features), 
                                                n_forests, num_trees))    # shape: (number of meaningful features, number of forests, number of trees)

    for i, feature in enumerate(most_important_features):
        for j, forest in enumerate(features_per_forest):    
            for k, tree in enumerate(forest):
                most_important_features_usage[i, j, k] = np.divide(usage_per_tree[j, k, i], len(tree)) 

    # Plotting the usage of `most_important_features` across isolation trees
    plt.figure(figsize=(12, 6))
    for i, feature in enumerate(most_important_features):
        plt.subplot(1, len(most_important_features), i+1)
        for j in range(n_forests):
            plt.bar(range(num_trees), most_important_features_usage[i, j, :], alpha=0.5, label=f'Forest {j+1}')
        plt.xlabel('Tree Index')
        plt.ylabel('Usage')
        plt.title(f'Usage of Feature {feature}')
        plt.legend()
    plt.tight_layout()

    # run.log({"most_important_features_usage_image": wandb.Image(plt)})

    # plt.show()

    return most_important_features_usage

In [ ]:
def average_usage(most_important_features_usage, seed_idx):
          
    thresholds = np.zeros((len(most_important_features_usage), n_forests))   # shape: (number of meaningful features, number of forests)

    for i, feature in enumerate(most_important_features_usage):
        for j, forest in enumerate(feature):    
            thresholds[i, j] = np.mean(forest)

    print(f"Thresholds shape: {thresholds.shape}")

    # data = [[f"Seed {seed_idx}", f"Forest {j+1}", thresholds[j]] for j in range(n_forests)]
    # columns = ["Seed", "Forest", "Average Usage"]
    # avg_usage_table = wandb.Table(data=data, columns=columns)
    # wandb.log({"Threshold Table": avg_usage_table})

    return  thresholds

In [ ]:
def majority_vote(most_important_features_usage, thresholds, fi_means):

    votes = np.zeros_like(most_important_features_usage)    # shape: (number of meaningful features, number of forests)
    results = np.zeros((most_important_features_usage.shape[1], most_important_features_usage.shape[2]), dtype=int)  # shape: (number of forests, number of trees)

    for i, feature in enumerate(most_important_features_usage):
        for j, forest in enumerate(feature):    
            votes[i, j] = forest < thresholds[i, j]

    print(f"Votes shape: {votes.shape}")

    for i in range(votes.shape[1]):         # for each forest
        for j in range(votes.shape[2]):     # for each tree
            weighted_votes = np.sum(np.multiply(votes[:, i, j], fi_means))  # weighted votes
            results[i, j] = 1 if weighted_votes > 0.5 else 0        # 1: tree to reject, 0: tree to accept

    print(f"Decisions shape: {results.shape}")
    # print(f"Decisions: {results}")

    return results

Now we are going to select the indexes of the trees to be removed from the original iForests

In [ ]:
def removing_trees(most_important_features_usage, thresholds, fi_means, iforests):
    """
    Remove the trees with usage percentage less than the threshold.
    """
    
    trees_to_remove = majority_vote(most_important_features_usage, thresholds, fi_means)

    # remove the trees from the forests
    for i, forest in enumerate(iforests):
        # print(f'Forest {i}:')
        
        # Get the list of trees and their corresponding features
        trees = forest.estimators_
        features = forest.estimators_features_

        # print(' Number of trees before removal:', len(trees))
        # print(' Number of features in trees before removal:', len(features))
        
        # Remove the specified trees and their features
        trees_to_keep = [tree for idx, tree in enumerate(trees) if trees_to_remove[i, idx] == 0]
        features_to_keep = [feature for idx, feature in enumerate(features) if trees_to_remove[i, idx] == 0]
        
        # Update the forest with the filtered lists
        forest.estimators_ = trees_to_keep
        forest.estimators_features_ = features_to_keep

        # Update internal attributes to match the reduced number of trees
        forest._decision_path_lengths = [forest._decision_path_lengths[idx] for idx in range(len(trees)) if trees_to_remove[i, idx] == 0]
        forest._average_path_length_per_tree = [forest._average_path_length_per_tree[idx] for idx in range(len(trees)) if trees_to_remove[i, idx] == 0]
        
        # print(' Number of trees after removal:', len(forest.estimators_))
        # print(' Number of features after removal:', len(forest.estimators_features_))

    print('Number of trees after removal: ', [len(iforests[i].estimators_) for i in range(len(iforests))])
    print('Number of features in trees after removal: ', [len(iforests[i].estimators_features_) for i in range(len(iforests))])
    print('Number of decision path lengths after removal: ', [len(iforests[i]._decision_path_lengths) for i in range(len(iforests))])
    print('Number of average path lengths after removal: ', [len(iforests[i]._average_path_length_per_tree) for i in range(len(iforests))])

This function computes the difference of the feature importance, for each features, between the original and new model

In [ ]:
def compute_difference(fi_means, new_fi_means, sorted_idx):
    """
    Compute the difference in feature importance between the original and new model.
    """
    
    diff = []
    for idx in sorted_idx:
        diff.append(new_fi_means[idx] - fi_means[idx])
        
    plt.figure(figsize=(10, 5))
    plt.bar(range(len(diff)), diff)
    plt.xticks(range(len(diff)), sorted_idx)
    plt.xlabel('Feature index')
    plt.ylabel('Difference in Feature Importance')
    plt.title('Difference in Feature Importance')
    # 
    wandb.log({"difference_feature_importance_image": wandb.Image(plt)})

    # plt.show()

In [ ]:
def gap_statistic_feature_selection(feature_importances, feature_names=None, plot=True):
    """
    Select features using the gap statistic method.

    Parameters:
    -----------
    feature_importances : dict or list
        If dict: {feature_name: importance_value}
        If list: [importance_values]
    feature_names : list, optional
        Feature names if feature_importances is a list
    plot : bool, default=True
        Whether to generate a plot of the sorted importances and gaps

    Returns:
    --------
    selected_features : list
        Names or indices of selected features
    threshold_gap : float
        The threshold gap used for selection
    all_gaps : list
        All computed gaps between consecutive features
    """
    # Convert input to dictionary if it's a list
    if isinstance(feature_importances, list):
        if feature_names is None:
            feature_names = [f"Feature {i}" for i in range(len(feature_importances))]
        feature_importances = {feature_names[i]: feature_importances[i] for i in range(len(feature_importances))}

    # Sort features by importance in descending order
    sorted_features = sorted(feature_importances.items(), key=lambda x: x[1], reverse=True)
    feature_names = [f[0] for f in sorted_features]
    importance_values = [f[1] for f in sorted_features]

    # Calculate gaps between consecutive sorted importance values
    gaps = [importance_values[i] - importance_values[i+1] for i in range(len(importance_values)-1)]

    # Find the largest gap and use it as threshold
    if gaps:
        max_gap_idx = np.argmax(gaps)
        threshold_gap = gaps[max_gap_idx]
        selected_features = feature_names[:max_gap_idx+1]
    else:
        threshold_gap = 0
        selected_features = feature_names

    # Generate plot if requested
    if plot:
        plt.figure(figsize=(12, 6))

        # Plot sorted feature importances
        plt.subplot(1, 2, 1)
        plt.bar(range(len(importance_values)), importance_values, color='skyblue')
        plt.xticks(range(len(feature_names)), feature_names, rotation=90)
        plt.ylabel('Feature Importance')
        plt.title('Sorted Feature Importances')

        # Highlight the cutoff point
        if gaps:
            plt.axvline(x=max_gap_idx + 0.5, color='red', linestyle='--',
                      label=f'Gap threshold: {threshold_gap:.3f}')
            plt.legend()

        # Plot gaps
        if gaps:
            plt.subplot(1, 2, 2)
            plt.bar(range(len(gaps)), gaps, color='lightgreen')
            plt.xticks(range(len(gaps)), [f"{feature_names[i]}-{feature_names[i+1]}" for i in range(len(gaps))], rotation=90)
            plt.ylabel('Gap Size')
            plt.title('Gaps Between Consecutive Features')

            # Highlight the maximum gap
            plt.bar(max_gap_idx, gaps[max_gap_idx], color='darkgreen')

        plt.tight_layout()
        wandb.log({"gap_statistic_feature_selection_image": wandb.Image(plt)})

        # plt.show()

    return selected_features, threshold_gap, gaps

## Launch experiments

In [ ]:
seeds = [0, 1, 2, 3, 4]
for seed in seeds:
    print(f"Seed: {seed}\n")

    # Load ODDS dataset
    for i, dataset in enumerate(odds_datasets.datasets_names):
        print(f"Dataset: {dataset}\n")

        X, y = odds_datasets.load(dataset)
        contamination = y.mean()
        # X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=test_size, random_state=seed, stratify=y)

        # X_tr, y_tr = shuffle(X_tr, y_tr, random_state=seed)
        X_tr, y_tr = shuffle(X, y, random_state=seed)
    

        # Init wandb to log the results
        run = wandb.init(
            project="diffi_optimization",
            name=f"experiment_{dataset}_forest_{n_forests}_trees_{num_trees}_seed_{seed}",
            config={
                "seed": seed,
                "dataset": dataset,
                "num_trees": num_trees,
                # Switched to automatic max_samples (see diffi_ranks_per_tree function)
                "max_samples": len(X_tr),
                "n_forests": n_forests,
                "contamination": contamination,
                "test_size": test_size,
                "threshold": "majority_vote",
            }
        )

        # TODO: compute also the average precision score
        sorted_idx, avg_f1, fi_means, fi_std, features_per_forest, fi_diffi_all, iforests, fi_diffi_inliers, fi_diffi_outliers = diffi_ranks_per_tree(
            X=X_tr, 
            y=y_tr, 
            n_trees=run.config.num_trees, 
            max_samples=run.config.max_samples, 
            n_iter=run.config.n_forests, 
            seed=seed,
            contamination=run.config.contamination
        )

        sorted_means = np.flip(np.argsort(fi_means))
        # [print(f"Feature {sorted_means[i]}: FI mean - {fi_means[sorted_means[i]]}, FI std - {fi_std[sorted_means[i]]}") for i in range(len(sorted_means))]
        # print('\n')

        fi_means_normalized = np.divide(fi_means, np.sum(fi_means))
        fi_std_normalized = np.divide(fi_std, np.sum(fi_std))

        print('Average F1 score: {:.4f}'.format(avg_f1))
        print('\n')

        num_forests = len(features_per_forest)
        num_trees = len(features_per_forest[0])

        log_model_config(run.config.contamination, run.config.max_samples)
        log_feature_importance(sorted_means, fi_means, fi_std, og_model=True)

        # Selecting the most important features using the gap statistic method
        feature_importances_dict = {sorted_means[i]: fi_means[sorted_means[i]] for i in range(len(sorted_means))}   # Convert to dictionary
        selected_features, gap_threshold, all_gaps = gap_statistic_feature_selection(feature_importances=feature_importances_dict)

        print(f"Most important features: {selected_features}")
        print(f"Gap threshold: {gap_threshold:.4f}")
        print("All gaps between consecutive features:")
        for i, gap in enumerate(all_gaps):
            feature_pair = [f"Feature {sorted_means[i]}", f"Feature {sorted_means[i+1]}"]
            print(f"  {feature_pair[0]} - {feature_pair[1]}: {gap:.4f}")

        log_feature_importance_heatmap(fi_diffi_inliers, for_inliers=True)
        log_feature_importance_heatmap(fi_diffi_outliers, for_inliers=False)

        _, usage_per_tree = feature_usage(features_per_forest=features_per_forest, features=sorted_means)

        most_important_features_usage = most_important_feature_usage(
            most_important_features=selected_features,
            usage_per_tree=usage_per_tree, 
            features_per_forest=features_per_forest
        )

        most_important_features_fi_means = fi_means_normalized[selected_features]
        print('Most important features FI means:', most_important_features_fi_means)
        print('\n')
        mif_fi_means_normalized = np.divide(most_important_features_fi_means, np.sum(most_important_features_fi_means))
        print('Most important features FI means normalized:', mif_fi_means_normalized)
        print('\n')
        thresholds = average_usage(most_important_features_usage, seed_idx=seed)
        print('Thresholds:', thresholds)
        print('\n')

        removing_trees(
            most_important_features_usage, 
            thresholds=thresholds,
            fi_means=mif_fi_means_normalized,
            iforests=iforests
        )

        print('\n')

        # TODO: compute also the average precision score
        new_sorted_idx, new_avg_f1, new_fi_means, new_fi_std, new_features_per_forest, new_fi_diffi_all, new_fi_diffi_inliers, new_fi_diffi_outliers = diffi_ranks_evaluation_only(
            X=X_tr, y=y_tr, iforest=iforests
        )

        print('New average F1 score: {:.4f}'.format(new_avg_f1))
        print('\n')

        new_sorted_means = np.flip(np.argsort(new_fi_means))
        log_feature_importance(new_sorted_means, new_fi_means, new_fi_std, og_model=False)

        # Log number of estimators in each forest of the new model
        data = [[f"Seed {seed}", f"Forest {i+1}", len(iforests[i].estimators_)] for i in range(len(iforests))]
        columns = ["Seed", "Forest", "Number of Estimators"]

        num_estimators_table = wandb.Table(data=data, columns=columns)
        run.log({"Number of Estimators Table in the new model": num_estimators_table})

        # Log the F1 scores
        data = [[f"Seed {seed}", num_forests, avg_f1, new_avg_f1]]
        columns = ["Seed", "Forest", "Original F1 Score", "New F1 Score"]
        avg_f1_scores_table = wandb.Table(data=data, columns=columns)
        run.log({"F1 Scores Table": avg_f1_scores_table})

        # Log the feature importance difference
        compute_difference(fi_means, new_fi_means, sorted_idx)

        print("---------------------------------------------------------------------------")

In [ ]:
run.finish()